# Hybrid imaging with time-varying image quality

This example shows how to set up an imaging environment with targets that have heterogeneous, time-varying rewards. The underlying environment accepts hybrid actions consisting of a target index and a desired acquisition delay. Here, a custom actor produces categorical logits over target/timing pairs, and a separate critic estimates the state value. Action wrappers convert the selected pair into a target index and delay in seconds. The final section demonstrates multi-satellite coordination through shared acquisition information and task intents. This example is part of an upcoming publication.


In [1]:
from collections.abc import Callable, Iterable
from typing import Any, ClassVar

import gymnasium as gym
import numpy as np
from Basilisk.architecture import bskLogging
from Basilisk.utilities import orbitalMotion
from scipy.interpolate import make_lsq_spline

from bsk_rl import ConstellationTasking, SatelliteTasking, act, obs, sats
from bsk_rl.data.unique_image_data import (
    GlobalReward,
    UniqueImageData,
    UniqueImageReward,
    UniqueImageStore,
)
from bsk_rl.scene.targets import CityTargets, Target
from bsk_rl.sim import dyn, fsw
from bsk_rl.utils.orbital import random_circular_orbit

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)

ALTITUDE_KM = 800.0
EARTH_RADIUS = orbitalMotion.REQ_EARTH * 1e3
ORBIT_PERIOD = (
    2
    * np.pi
    * np.sqrt((orbitalMotion.REQ_EARTH + ALTITUDE_KM) ** 3 / orbitalMotion.MU_EARTH)
)
N_AHEAD = 40
SIM_RATE = 0.5
IMAGING_FRACTIONS = (0.0, 0.5, 1.0)
SPLINE_DEGREE = 2
SPLINE_KNOTS = np.r_[np.zeros(3), [1 / 3, 2 / 3], np.ones(3)]

/Users/lorenzzo/Local/repo_updates/issue_363/.venv_issue_363/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Targets and quality

Acquisition quality can depend on viewing geometry or a stakeholder preference for the acquisition time. This example includes the paper's off-nadir and periodic quality models. The distance and glare models are omitted to keep the example compact.

`QualityTargets` extends `CityTargets` by assigning a quality model and weight to each target. Setting `quality_weight=1.0` makes reward equal to priority times quality, while `quality_weight=0.0` makes reward depend only on priority. Setting `quality_weight=(0.0, 1.0)` samples a separate weight for each target on every reset. Choose a single model with `quality_types=("quality_off_nadir_angle",)` or `quality_types=("quality_periodic",)`, or sample uniformly from `quality_types=("quality_off_nadir_angle", "quality_periodic")`. The absolute acquisition reward is

$$r_i(t)=p_i\big[1-w_{q,i}+w_{q,i}q_i(t)\big],$$

where $p_i$ is target priority, $w_{q,i}$ is its quality weight, and $q_i(t)\in[0,1]$ is the quality at acquisition time. The rewarder below determines whether an acquisition receives this absolute reward or only its improvement over the best completed reward.

Off-nadir quality uses the angle between the target's local vertical and its viewing direction: $q=\mathrm{clip}(1-2\theta/\pi,0,1)$. Future samples use predicted geometric viewing directions; at the current time, realized instrument pointing is used. Periodic quality represents a known, time-varying acquisition preference, using one or two sinusoidal components with periods between 150 and 570 seconds. Frequencies in the code are angular frequencies in radians per second. These are illustrative acquisition-quality models, not a radiometric sensor simulation.


In [2]:
def periodic_quality_function(t, amplitudes, frequencies, phases):
    """Evaluate a window-independent sum of periodic components.

    ``frequencies`` are angular frequencies in radians per second.
    """
    t = np.asarray(t, dtype=float)
    amplitudes = np.asarray(amplitudes, dtype=float)
    frequencies = np.asarray(frequencies, dtype=float)
    phases = np.asarray(phases, dtype=float)

    if amplitudes.ndim != 1 or frequencies.ndim != 1 or phases.ndim != 1:
        raise ValueError("amplitudes, frequencies, and phases must be 1-D arrays.")
    if not (len(amplitudes) == len(frequencies) == len(phases)):
        raise ValueError(
            "amplitudes, frequencies, and phases must have the same length."
        )
    if len(amplitudes) == 0:
        raise ValueError("At least one periodic component is required.")
    if not (
        np.all(np.isfinite(t))
        and np.all(np.isfinite(amplitudes))
        and np.all(np.isfinite(frequencies))
        and np.all(np.isfinite(phases))
    ):
        raise ValueError("Periodic-function inputs must all be finite.")
    if np.any(frequencies <= 0):
        raise ValueError("Periodic frequencies must be greater than zero.")

    amplitude_sum = np.sum(np.abs(amplitudes))
    if amplitude_sum <= 0:
        raise ValueError("At least one periodic amplitude must be nonzero.")

    quality = np.sum(
        np.array(
            [
                amplitudes[i] * np.sin(frequencies[i] * t + phases[i])
                for i in range(len(amplitudes))
            ]
        ),
        axis=0,
    )

    quality = (quality + amplitude_sum) / (2 * amplitude_sum)

    return quality.item() if np.ndim(quality) == 0 else quality


def quality_periodic(sat, tgt, t, amplitudes, frequencies, phases, **kwargs):

    return periodic_quality_function(
        t,
        amplitudes=amplitudes,
        frequencies=frequencies,
        phases=phases,
    )


def _positions_at_times(sat, times, trajectory_r_BP_P=None):
    """Return predicted positions, using dynamics for the realized current time."""
    times = np.asarray(times, dtype=float)
    current_mask = np.abs(times - sat.simulator.sim_time) <= 1e-6

    if times.ndim == 0:
        if current_mask:
            return np.asarray(sat.dynamics.r_BN_P, dtype=float)
    elif np.all(current_mask):
        return np.broadcast_to(
            np.asarray(sat.dynamics.r_BN_P, dtype=float),
            times.shape + (3,),
        ).copy()

    if trajectory_r_BP_P is None:
        trajectory_r_BP_P = sat.trajectory.r_BP_P
    positions = np.asarray(trajectory_r_BP_P(times), dtype=float)

    if times.ndim > 0 and np.any(current_mask):
        positions = positions.copy()
        positions[current_mask] = sat.dynamics.r_BN_P
    return positions


def quality_off_nadir_angle(sat, tgt, t, trajectory_r_BP_P=None, **kwargs):
    """Return quality based on the target-centered instrument viewing angle.

    At the current simulation time, use the realized instrument direction so
    the reward reflects the actual imaging attitude. For future profile samples,
    where the attitude is not yet known, use the geometric target-to-satellite
    direction. Quality decreases linearly from one for a target directly below
    the instrument to zero for a target viewed at its local horizon.
    """
    t = np.asarray(t, dtype=float)
    scalar_input = t.ndim == 0
    r_BN_P = _positions_at_times(sat, t, trajectory_r_BP_P)

    target_position = np.asarray(tgt.r_LP_P, dtype=float)
    target_radius = np.linalg.norm(target_position)
    target_to_camera = r_BN_P - target_position
    view_norm = np.linalg.norm(target_to_camera, axis=-1, keepdims=True)
    if (
        np.any(~np.isfinite(r_BN_P))
        or not np.all(np.isfinite(target_position))
        or target_radius <= 0.0
        or np.any(view_norm <= 0.0)
    ):
        raise ValueError("Off-nadir geometry must contain finite, nonzero vectors.")

    local_vertical_hat_P = target_position / target_radius
    view_hat_P = target_to_camera / view_norm

    realized_mask = np.abs(t - sat.simulator.sim_time) <= 1e-6
    if np.any(realized_mask):
        c_hat_P = np.asarray(sat.fsw.c_hat_P, dtype=float)
        if c_hat_P.shape != (3,) or not np.all(np.isfinite(c_hat_P)):
            raise ValueError("c_hat_P must be a finite three-component vector.")
        pointing_norm = np.linalg.norm(c_hat_P)
        if pointing_norm <= 0.0:
            raise ValueError("c_hat_P must be nonzero.")
        if scalar_input:
            view_hat_P = -c_hat_P / pointing_norm
        else:
            view_hat_P = view_hat_P.copy()
            view_hat_P[realized_mask] = -c_hat_P / pointing_norm

    off_nadir = np.arccos(
        np.clip(np.sum(view_hat_P * local_vertical_hat_P, axis=-1), -1.0, 1.0)
    )

    quality = np.clip(1 - off_nadir / (np.pi / 2), 0.0, 1.0)
    return quality.item() if scalar_input else quality


QUALITY_FUNCTIONS = {
    "quality_off_nadir_angle": quality_off_nadir_angle,
    "quality_periodic": quality_periodic,
}


def get_randomized_quality_function_params(quality_function_name):
    if quality_function_name in ["quality_off_nadir_angle"]:
        return {}
    elif quality_function_name == "quality_periodic":
        n_components = np.random.randint(1, 3)
        amplitudes = np.random.uniform(0.1, 1.0, size=n_components)
        frequencies = (
            2
            * np.pi
            * np.random.uniform(
                10.0 / 5700.0,
                1.0 / 150.0,
                size=n_components,
            )
        )
        phases = np.random.uniform(0.0, 2 * np.pi, size=n_components)
        return {
            "amplitudes": amplitudes,
            "frequencies": frequencies,
            "phases": phases,
        }
    else:
        raise ValueError(f"Unknown quality function: {quality_function_name}")

### Assign quality models to targets

`QualityTargets` extends `CityTargets`: it generates city-based targets through the parent class and then attaches a quality weight, a selected quality model, and that model's randomized parameters to each target. A scalar weight applies to all targets; a two-element range samples a separate weight for each target. A one-element tuple of quality names selects the same model for every target.

On each reset, the scenario initializes the best obtained reward to `0.0` and clears the per-satellite capture timestamps. These fields support marginal rewards for re-imaging and the cooldown filter used in the coordination example below.


In [3]:
from copy import deepcopy


class QualityTargets(CityTargets):
    """City-based target scenario with per-target quality models and weights."""

    def __init__(
        self,
        n_targets: int | tuple[int, int],
        n_select_from: int | None = None,
        location_offset: float = 0,
        priority_distribution: Callable | None = None,
        radius: float = orbitalMotion.REQ_EARTH * 1e3,
        quality_weights: float | list[float] = 0.5,
        quality_function_name: list[str] | None = None,
        quality_function_weights: None | list[float] = None,
    ) -> None:

        super().__init__(
            n_targets, n_select_from, location_offset, priority_distribution, radius
        )
        self.quality_weights = quality_weights
        if quality_function_name is None:
            self.quality_function_name = [
                "quality_off_nadir_angle",
                "quality_periodic",
            ]
        self.quality_function_name = deepcopy(quality_function_name)
        self.quality_function_weights = quality_function_weights
        if self.quality_function_weights is not None:
            assert len(self.quality_function_name) == len(quality_function_weights), (
                "quality_function_weights must match quality_functions"
            )
            self.quality_function_weights = np.array(quality_function_weights) / np.sum(
                np.array(quality_function_weights)
            )

    def regenerate_targets(self) -> None:
        super().regenerate_targets()

        for tgt in self.targets:
            if isinstance(self.quality_weights, (list, tuple, np.ndarray)):
                quality_weight = np.random.uniform(
                    self.quality_weights[0], self.quality_weights[1]
                )
            else:
                quality_weight = self.quality_weights

            quality_function = np.random.choice(
                self.quality_function_name, p=self.quality_function_weights
            )

            quality_function_params = get_randomized_quality_function_params(
                quality_function
            )

            tgt.quality_weight = quality_weight
            tgt.quality_function_name = quality_function
            tgt.quality_function_params = quality_function_params
            tgt.highest_quality_obtained = None
            tgt.highest_quality_time = None
            tgt.highest_reward_obtained = 0.0
            tgt.last_img_time_per_sat = {}

## Rewarding physical captures

`QualityImageStore` detects a completed acquisition from an increase in stored instrument data and evaluates quality using the target's assigned model and the satellite state at capture time. Requested acquisition time and realized capture time are recorded separately because pointing constraints can delay capture.

`QualityImageReward` converts these acquisition events into step rewards. With `incremental_score=False`, a target is rewarded only for its first completed acquisition. With `incremental_score=True`, a capture earns $\max(r_i(t)-r_{i,\mathrm{max}},0)$, where $r_{i,\mathrm{max}}$ is the best absolute acquisition reward completed before that capture. Pending intents affect predicted marginal profiles, but do not increase the completed-reward baseline used for actual reward allocation.


In [4]:
class QualityImageData(UniqueImageData):
    def __init__(self, imaged=None, duplicates=0, known=None, image_events=None):
        super().__init__(imaged=imaged, duplicates=duplicates, known=known)

        if image_events is None:
            image_events = []
        self.image_events = [event.copy() for event in image_events]

    def __add__(self, other: "QualityImageData") -> "QualityImageData":
        imaged = self.imaged | other.imaged
        image_events = self.image_events + other.image_events
        duplicates = (
            self.duplicates
            + other.duplicates
            + len(self.imaged)
            + len(other.imaged)
            - len(imaged)
        )
        known = self.known | other.known
        return self.__class__(
            imaged=imaged,
            duplicates=duplicates,
            known=known,
            image_events=image_events,
        )


def get_quality(sat, tgt, t=None, **function_context):
    """Evaluate the quality function assigned to a target."""
    if t is None:
        t = sat.simulator.sim_time

    quality_function = tgt.quality_function_name
    quality_function_params = {
        **tgt.quality_function_params,
        **function_context,
    }
    return QUALITY_FUNCTIONS[quality_function](
        sat,
        tgt,
        t=t,
        **quality_function_params,
    )


class QualityImageStore(UniqueImageStore):
    """Record quality-aware target acquisitions, including repeated captures."""

    data_type = QualityImageData

    def __init__(
        self,
        *args,
        record_image_profiles: bool = False,
        image_profile_samples: int = 25,
        **kwargs,
    ) -> None:
        """Initialize image-quality logging options.

        Args:
            record_image_profiles: Sample quality over the acquisition access window.
                This is intended for post-test visualization and is disabled by default
                to avoid adding work to training environments.
            image_profile_samples: Number of evenly spaced samples in each profile.
        """
        if image_profile_samples < 2:
            raise ValueError("image_profile_samples must be at least 2")
        self.record_image_profiles = record_image_profiles
        self.image_profile_samples = image_profile_samples
        super().__init__(*args, **kwargs)

    def _access_window(self, target: "Target", time: float) -> tuple[float, float]:
        """Return a containing window, otherwise the nearest known window or NaNs."""
        candidates = []
        for opportunity in getattr(self.satellite, "opportunities", []):
            if opportunity.get("type") != "target":
                continue
            opportunity_target = opportunity.get("object")
            if opportunity_target is not target and getattr(
                opportunity_target, "id", None
            ) != getattr(target, "id", None):
                continue
            window = opportunity.get("window")
            if window is None or len(window) != 2:
                continue
            window_open, window_close = map(float, window)
            if window_open <= time <= window_close:
                return window_open, window_close
            candidates.append((window_open, window_close))

        if candidates:
            return min(
                candidates,
                key=lambda window: min(
                    abs(time - window[0]),
                    abs(time - window[1]),
                ),
            )
        return np.nan, np.nan

    def _requested_imaging_time(self) -> float:
        """Read the controller's absolute requested imaging time, when available."""
        try:
            return float(self.satellite.fsw.insControl.imagingTime) * 1e-9
        except (AttributeError, TypeError, ValueError):
            return np.nan

    def _quality_profile(
        self,
        target: "Target",
        window_open: float,
        window_close: float,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Sample raw target quality over an access window."""
        if (
            not self.record_image_profiles
            or not np.all(np.isfinite([window_open, window_close]))
            or window_close <= window_open
            or not all(
                hasattr(target, attribute)
                for attribute in (
                    "quality_function_name",
                    "quality_function_params",
                )
            )
        ):
            return np.asarray([], dtype=float), np.asarray([], dtype=float)

        profile_times = np.linspace(
            window_open,
            window_close,
            self.image_profile_samples,
        )

        profile_qualities = np.asarray(
            get_quality(
                self.satellite,
                target,
                t=profile_times,
                w_o=window_open,
                w_c=window_close,
            ),
            dtype=float,
        )

        if profile_qualities.shape != profile_times.shape or not np.all(
            np.isfinite(profile_qualities)
        ):
            return np.asarray([], dtype=float), np.asarray([], dtype=float)
        return profile_times, np.clip(profile_qualities, 0.0, 1.0)

    def get_log_state(self) -> float:
        """Log the instantaneous storage unit state at the end of each step.
        Returns:
            float: storedData from satellite storage unit
        """
        msg = self.satellite.dynamics.storageUnit.storageUnitDataOutMsg.read()

        return msg.storedData[0]

    def compare_log_states(
        self, old_state: float, new_state: float
    ) -> QualityImageData:
        """Record an acquisition when stored instrument data increases.

        Args:
            old_state: Previous scalar stored-data reading.
            new_state: Current scalar stored-data reading.

        Returns:
            QualityImageData containing the captured target and acquisition event,
            or an empty record when storage does not increase. Repeated captures
            of the same target are recorded as separate events.
        """
        data_increase = new_state - old_state

        if data_increase <= 0:
            return QualityImageData()
        else:
            assert self.satellite.latest_target is not None
            target = self.satellite.latest_target
            self.update_target_colors([target])
            image_time = float(self.satellite.simulator.sim_time)
            window_open, window_close = self._access_window(target, image_time)

            quality = get_quality(
                self.satellite,
                target,
                t=image_time,
                w_o=window_open,
                w_c=window_close,
            )

            target.last_quality_obtained = quality
            target.last_quality_time = image_time
            if (
                target.highest_quality_obtained is None
                or quality > target.highest_quality_obtained
            ):
                target.highest_quality_obtained = quality
                target.highest_quality_time = image_time

            target.last_img_time_per_sat[self.satellite.name] = image_time

            profile_times, profile_qualities = self._quality_profile(
                target,
                window_open,
                window_close,
            )
            image_event = {
                "time": image_time,
                "requested_time": self._requested_imaging_time(),
                "target_id": str(target.id),
                "target_name": str(target.name),
                "satellite_name": str(self.satellite.name),
                "priority": float(target.priority),
                "quality": float(quality),
                "quality_function_name": str(
                    getattr(target, "quality_function_name", "legacy_distance")
                ),
                "quality_function_params": dict(
                    getattr(target, "quality_function_params", {})
                ),
                "access_window_open": window_open,
                "access_window_close": window_close,
                "imaging_time_fraction": (image_time - window_open)
                / (window_close - window_open),
                "profile_times": profile_times,
                "profile_qualities": profile_qualities,
                "quality_weight": np.nan,
                "reward": 0.0,
                "rewarded": False,
            }
            return QualityImageData(
                imaged={target},
                image_events=[image_event],
            )


class QualityImageReward(UniqueImageReward):
    """Reward initial captures or positive re-imaging improvements."""

    data_store_type = QualityImageStore

    def __init__(
        self,
        reward_fn: Callable,
        record_image_profiles: bool = False,
        image_profile_samples: int = 25,
        incremental_score: bool = False,
    ):
        """Initialize the rewarder.

        Args:
            reward_fn: Callback evaluated as
                reward_fn(priority, quality, quality_weight=target.quality_weight).
                The quality weight is supplied by each target.
            record_image_profiles: Sample target quality over each successful image's
                access window for post-test visualization.
            image_profile_samples: Number of samples stored in each quality profile.
            incremental_score: If True, reward only positive improvements over
                the best completed reward for each target. If False, reward only
                the first completed acquisition of each target.
        """

        super().__init__(reward_fn)
        self.data_store_kwargs.update(
            record_image_profiles=record_image_profiles,
            image_profile_samples=image_profile_samples,
        )
        self.incremental_score = incremental_score

    def create_data_store(self, satellite) -> None:
        """Create the store, bypassing the unique-target filter for re-imaging."""
        if self.incremental_score:
            GlobalReward.create_data_store(self, satellite)
        else:
            super().create_data_store(satellite)

    def _update_event_reward_info(
        self,
        event: dict[str, Any],
        target: "Target",
        quality_weight: float,
        reward: float,
        rewarded: bool,
    ) -> None:
        """Attach reward information to an acquisition event in place."""
        event["quality_weight"] = float(quality_weight)
        event["reward"] = float(reward)
        event["rewarded"] = rewarded

        profile_qualities = np.asarray(event["profile_qualities"], dtype=float)
        if profile_qualities.size == 0:
            event["profile_rewards"] = np.asarray([], dtype=float)
            return

        try:
            profile_rewards = np.asarray(
                self.reward_fn(
                    target.priority,
                    profile_qualities,
                    quality_weight=quality_weight,
                ),
                dtype=float,
            )
            if profile_rewards.ndim == 0:
                profile_rewards = np.full_like(profile_qualities, profile_rewards)
        except (TypeError, ValueError):
            profile_rewards = np.asarray(
                [
                    self.reward_fn(
                        target.priority,
                        quality,
                        quality_weight=quality_weight,
                    )
                    for quality in profile_qualities
                ],
                dtype=float,
            )
        event["profile_rewards"] = profile_rewards

    def calculate_reward(
        self, new_data_dict: dict[str, QualityImageData]
    ) -> dict[str, float]:
        """Allocate step rewards for completed acquisitions.

        Evaluate reward_fn(priority, quality, quality_weight=target.quality_weight).
        Nonincremental scoring rewards only the first acquisition of a target;
        incremental scoring rewards positive improvements over its best completed
        reward. For simultaneous captures, allocate reward among satellites tied
        for the highest realized quality and give the others zero.

        Args:
            new_data_dict: Acquisition data keyed by satellite name.

        Returns:
            Dictionary mapping each satellite name to its total reward this step.
        """

        event_per_target = {}
        targets_by_id = {}
        for sat_id, new_data in new_data_dict.items():
            targets_by_id.update({str(target.id): target for target in new_data.imaged})
            for event in new_data.image_events:
                target_id = event["target_id"]
                if target_id not in event_per_target:
                    event_per_target[target_id] = []
                event_per_target[target_id].append((sat_id, event))

        reward = {sat_id: 0.0 for sat_id in new_data_dict}
        for tgt_id, events in event_per_target.items():
            target = targets_by_id.get(tgt_id)

            if target in self.data.imaged and not self.incremental_score:
                for sat_id, event in events:
                    self._update_event_reward_info(
                        event,
                        target,
                        target.quality_weight,
                        reward=0.0,
                        rewarded=False,
                    )
                continue

            time_list = np.array([event["time"] for _, event in events])
            time_events = set(time_list)
            time_events_sorted = sorted(time_events)

            for time_event in time_events_sorted:
                events_at_time = [
                    (sat_id, event)
                    for sat_id, event in events
                    if event["time"] == time_event
                ]

                max_quality = max(event["quality"] for _, event in events_at_time)
                sat_ids_max_quality = [
                    sat_id
                    for sat_id, event in events_at_time
                    if event["quality"] == max_quality
                ]

                scaling_conflict_score = {
                    sat_id: (
                        1.0 / len(sat_ids_max_quality)
                        if sat_id in sat_ids_max_quality
                        else 0.0
                    )
                    for sat_id, _ in events_at_time
                }

                if self.incremental_score:
                    if target.highest_reward_obtained is None:
                        highest_reward_obtained = 0
                    else:
                        highest_reward_obtained = target.highest_reward_obtained

                highest_new_reward = None
                for sat_id, event in events_at_time:
                    quality = event.get("quality", 0.0)

                    step_reward = self.reward_fn(
                        target.priority, quality, quality_weight=target.quality_weight
                    )

                    if self.incremental_score:
                        if (
                            highest_new_reward is None
                            or step_reward > highest_new_reward
                        ):
                            highest_new_reward = step_reward
                        step_reward = max(0.0, step_reward - highest_reward_obtained)

                    step_reward = step_reward * scaling_conflict_score.get(sat_id, 0.0)

                    reward[sat_id] += step_reward

                    self._update_event_reward_info(
                        event,
                        target,
                        target.quality_weight,
                        step_reward,
                        rewarded=step_reward > 0.0,
                    )

                if (
                    self.incremental_score
                    and highest_new_reward > highest_reward_obtained
                ):
                    target.highest_reward_obtained = highest_new_reward
                    target.highest_reward_time = time_event

        return reward

## Satellite and observations

Following the paper's observation representation, a quadratic B-spline approximates the estimated marginal reward over each remaining observation window. Its five coefficients are passed to the agent.

The observation has 32 satellite/context features (12 spacecraft properties and 20 priority-density bins), followed by 40 target tokens with 12 features each: five reward-spline coefficients, relative position (3), pointing angle, angular rate, and opening and closing offsets. Both the spline and action wrapper parameterize the **remaining** window, using the current time as the start when the window is already open. The action wrapper enforces a minimum delay of 0.5 seconds. Opening offsets in the observation remain signed and can therefore be negative for an already-open window.

`PriorityDensity` is a custom observation class that summarizes future imaging opportunities in 20 consecutive 180-second bins. Each target contributes its priority, divided by five, to the bin containing the closing time of its next available window. This provides context about upcoming opportunities beyond the 40 targets represented individually.

`TargetBookkeeping` stores the best completed reward known for each target and, when the communication mode uses them, competing intended rewards. Its combined baseline is the larger of the completed and intended values. An intent changes the expected marginal benefit of an acquisition; it does not count as a completed image or increase the obtained reward.

`estimate_quality_profile` samples absolute predicted reward at 25 times over the remaining window, subtracts the satellite's current bookkeeping baseline, and clips the result to [0, 1] before fitting the spline. With empty bookkeeping, the baseline is zero. Absolute predictions are cached, but the baseline is read on every call. The coordination loop updates bookkeeping and clears the observation cache before requesting a new observation, so the spline values change while the observation dimensions remain fixed.

In [5]:
class PriorityDensity(obs.Observation):
    """Priority sums in twenty future 180-second opportunity-closing bins."""

    def __init__(self):
        super().__init__(name="priority_density")

    def get_obs(self):
        now = self.simulator.sim_time
        self.satellite.calculate_additional_windows(
            now + 21 * 180 - self.satellite.window_calculation_time
        )
        upcoming = self.satellite.upcoming_opportunities_dict(types="target")
        density = np.zeros(20)
        for target, windows in upcoming.items():
            index = int(np.floor((windows[0][1] - now) / 180))
            if 0 <= index < len(density):
                density[index] += target.priority / 5.0
        return density


class TargetBookkeeping:
    def __init__(self):
        self._reset()

    def _reset(self):
        self.bookkeeping = {}
        self.bookkeeping_imaged = {}
        self.bookkeeping_intended = {}

    def update(
        self,
        tgt_id: str,
        obtained_reward: float,
        time: float,
    ) -> None:
        if tgt_id not in self.bookkeeping_imaged:
            self.bookkeeping_imaged[tgt_id] = {"reward": obtained_reward, "time": time}

        else:
            if obtained_reward > self.bookkeeping_imaged[tgt_id]["reward"]:
                self.bookkeeping_imaged[tgt_id] = {
                    "reward": obtained_reward,
                    "time": time,
                }

        self._update()

    def update_intended(
        self,
        tgt_id: str,
        expected_reward: float,
        time: float,
    ) -> None:

        if (
            tgt_id in self.bookkeeping_imaged
            and expected_reward <= self.bookkeeping_imaged[tgt_id]["reward"]
        ):
            return

        if tgt_id not in self.bookkeeping_intended:
            self.bookkeeping_intended[tgt_id] = {
                "reward": expected_reward,
                "time": time,
            }

        else:
            if expected_reward > self.bookkeeping_intended[tgt_id]["reward"]:
                self.bookkeeping_intended[tgt_id] = {
                    "reward": expected_reward,
                    "time": time,
                }

        self._update()

    def delete_intended(self):
        self.bookkeeping_intended = {}
        self._update()

    def _update(self):
        self.bookkeeping = copy(self.bookkeeping_imaged)
        tgt_ids_imaged = set(self.bookkeeping_imaged.keys())
        tgt_ids_intended = set(self.bookkeeping_intended.keys())
        for tgt_id in tgt_ids_intended:
            if tgt_id not in tgt_ids_imaged:
                self.bookkeeping[tgt_id] = self.bookkeeping_intended[tgt_id]
            else:
                if (
                    self.bookkeeping_intended[tgt_id]["reward"]
                    > self.bookkeeping_imaged[tgt_id]["reward"]
                ):
                    self.bookkeeping[tgt_id] = self.bookkeeping_intended[tgt_id]

    def get_reward(self, tgt_id: str) -> float:
        if tgt_id not in self.bookkeeping:
            return 0.0
        else:
            return self.bookkeeping[tgt_id]["reward"]


def _quality_profile_context(sat):
    """Cache trajectory data shared by all quality profiles at one simulation step."""
    trajectory = sat.trajectory
    cache_key = (
        id(trajectory),
        sat.simulator.sim_time,
        trajectory.sim_time,
        len(trajectory.times),
    )
    cache = getattr(sat, "_quality_profile_context", None)

    if cache is None or cache["key"] != cache_key:
        cache = {
            "key": cache_key,
            "trajectory_r_BP_P": trajectory.r_BP_P,
            "full_reward_profiles": {},
        }
        sat._quality_profile_context = cache

    return cache


def estimate_absolute_reward(sat, tgt, times):
    """Predict acquisition reward without completed or intended reward baselines.

    ``times`` contains absolute simulator times. Use the same trajectory and
    quality assumptions as the observation profiles, but do not infer reward
    from their clipped marginal splines.
    """
    profile_context = _quality_profile_context(sat)
    raw_quality = QUALITY_FUNCTIONS[tgt.quality_function_name](
        sat,
        tgt,
        np.asarray(times, dtype=float),
        **tgt.quality_function_params,
        trajectory_r_BP_P=profile_context["trajectory_r_BP_P"],
    )
    return np.asarray(
        reward_function(tgt.priority, raw_quality, quality_weight=tgt.quality_weight),
        dtype=float,
    )


def estimate_quality_profile(sat, opp, return_rewards: bool = False):
    tgt = opp["object"]
    window = opp["window"]
    window_open = np.max([window[0], sat.simulator.sim_time])
    window_close = window[1]
    profile_context = _quality_profile_context(sat)

    profile_key = (
        str(tgt.id),
        float(window_open),
        float(window_close),
    )
    profile = profile_context["full_reward_profiles"].get(profile_key)

    if profile is None:
        times = np.linspace(window_open, window_close, 25)
        full_reward = estimate_absolute_reward(sat, tgt, times)
        profile = {
            "times": times,
            "full_reward": full_reward,
        }
        profile_context["full_reward_profiles"][profile_key] = profile

    obtained_reward = (
        sat.bookkeeping.get_reward(tgt.id) if hasattr(sat, "bookkeeping") else 0.0
    )

    marginal_reward = np.clip(
        profile["full_reward"] - obtained_reward,
        a_min=0.0,
        a_max=1.0,
    )

    if return_rewards:
        return marginal_reward

    # Normalize sample times to [0, 1] over the remaining access window.
    x_vector = (profile["times"] - window_open) / (window_close - window_open)

    spline = make_lsq_spline(x_vector, marginal_reward, t=SPLINE_KNOTS, k=SPLINE_DEGREE)

    return spline.c

### Define the satellite

`QualitySatellite` combines the spacecraft dynamics and flight software with the observations defined above. Its observation specification includes spacecraft properties, `PriorityDensity`, and the next `N_AHEAD` target opportunities. The opportunity callback uses `estimate_quality_profile` to supply the five reward-spline coefficients alongside each target's geometry and window offsets. Each satellite has its own bookkeeping, which is cleared when the environment resets.

The action specification uses `act.Image(n_ahead_image=N_AHEAD, hybrid=True)`. This enables a native imaging action containing a target-opportunity index and a desired acquisition delay in seconds. The wrappers below expose normalized timing fractions and, optionally, a categorical action over target/timing pairs. `satellite_args` supplies the orbit, pointing constraints, timing tolerances, and spacecraft resource settings shared by training and evaluation.


In [6]:
from functools import partial


class QualitySatellite(sats.ImagingSatellite):
    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = fsw.SteeringImagerFSWModel
    observation_spec: ClassVar = [
        obs.SatProperties(
            dict(prop="omega_BN_B", norm=0.03),
            dict(prop="c_hat_H"),
            dict(prop="r_BN_P", norm=EARTH_RADIUS),
            dict(prop="v_BN_P", norm=7616.5),
        ),
        PriorityDensity(),
        obs.OpportunityProperties(
            dict(fn=partial(estimate_quality_profile)),
            dict(prop="r_LB_H", norm=EARTH_RADIUS),
            dict(prop="target_angle", norm=np.pi / 2),
            dict(prop="target_angle_rate", norm=0.03),
            dict(prop="opportunity_open", norm=300),
            dict(prop="opportunity_close", norm=300),
            n_ahead_observe=N_AHEAD,
        ),
    ]
    action_spec: ClassVar = [act.Image(n_ahead_image=N_AHEAD, hybrid=True)]

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.bookkeeping = TargetBookkeeping()

    def reset_pre_sim_init(self) -> None:
        super().reset_pre_sim_init()
        self.bookkeeping._reset()


def satellite_args():
    return QualitySatellite.default_sat_args(
        oe=lambda: random_circular_orbit(alt=ALTITUDE_KM, i=45),
        imageAttErrorRequirement=0.01,
        imageRateErrorRequirement=0.01,
        imageTargetMinimumElevation=np.arctan(800 / 500),
        timeToleranceLower=1.0,  # Seconds; BSK-RL converts to nanoseconds.
        timeToleranceUpper=0.0,  # Zero disables the upper bound, allowing late capture.
        u_max=0.4,
        K1=0.25,
        K3=3.0,
        servo_P=30,
        omega_max=np.radians(5),
        rwBasePower=20,
        disturbance_vector=np.zeros(3),
        batteryStorageCapacity=80 * 3600 * 2 * 1000,
        storedCharge_Init=0.99 * 80 * 3600 * 2 * 1000,
        maxWheelSpeed=15000,
        wheelSpeeds=lambda: np.random.uniform(-1, 1, 3),
        dataStorageCapacity=2 * 8e6 * 25 * 1000,
        storageInit=0,
        transmitterPacketSize=0,
    )

## Minimal wrappers and environment factory

With `hybrid=True`, the native imaging action accepts a target index and a desired acquisition delay in seconds. This delay specifies when to try to acquire the image; it is not the exposure duration. The wrappers let a policy select a target and a normalized position within its remaining observation window.

### From a discrete option to a target and timing fraction

Let $N$ be the number of upcoming target opportunities and let $F=(F_0,\ldots,F_{D-1})$ contain $D$ predefined timing fractions. The policy selects one categorical action $a\in\{0,\ldots,ND-1\}$. `WrapperDiscreteToHybridAction` decodes it as

$$
i=\left\lfloor\frac{a}{D}\right\rfloor,\qquad
k=a\bmod D,\qquad f=F_k.
$$

Here $i$ is the zero-based target-opportunity index and $k$ is the timing-option index. The inverse mapping is $a=iD+k$. Timing options vary fastest within each target. With `N_AHEAD=40` and `IMAGING_FRACTIONS=(0.0, 0.5, 1.0)`, each satellite has `Discrete(120)`:

| Discrete action | Target index | Timing fraction |
|---|---|---|
| 6 | 2 | 0.0 |
| 7 | 2 | 0.5 |
| 8 | 2 | 1.0 |

The actor therefore needs a categorical distribution over 120 choices. Although these choices are translated into real-valued delays, the policy only chooses among the three listed fractions; it does not output an arbitrary continuous timing value.

### From a timing fraction to seconds

For current simulation time $t_{\mathrm{now}}$ and the selected opportunity's physical window $[t_{\mathrm{open}},t_{\mathrm{close}}]$, define the remaining opening and closing offsets

$$
u=\max(0,t_{\mathrm{open}}-t_{\mathrm{now}}),\qquad
v=t_{\mathrm{close}}-t_{\mathrm{now}}.
$$

`WrapperActionDuration` clips $f$ to $[0,1]$ and returns

$$
\Delta t_{\mathrm{img}}=\max\bigl(0.5,\;u+f(v-u)\bigr),\qquad
t_d=t_{\mathrm{now}}+\Delta t_{\mathrm{img}}.
$$

The native action receives `(i, delta_t_img)` and the controller uses the absolute desired acquisition time $t_d$. For example, if the current time is 100 seconds and target opportunity 2 spans [120, 180] seconds, action 7 selects the midpoint: a 50-second delay and a desired acquisition time of 150 seconds. If the window instead spans [90, 140] seconds, the same fraction selects the midpoint of the remaining window: a 20-second delay and a desired acquisition time of 120 seconds. Fraction zero requests the earliest remaining time, subject to the 0.5-second minimum delay.

The wrapper stack processes actions in this order:

```text
discrete action a
  -> WrapperDiscreteToHybridAction: (target index i, fraction f)
  -> WrapperActionDuration: (target index i, delay in seconds)
  -> WrapperActionLogging
  -> native imaging action: absolute desired time = now + delay
```

For a constellation, this conversion is applied separately to each supplied entry in the satellite-name action dictionary.

### Acquisition tolerances and action limits

Acquisition also requires the pointing-angle and angular-rate constraints to be satisfied. When both timing bounds are enabled, the allowed acquisition times satisfy

$$
t_d-\tau_{\mathrm{lower}}\leq t\leq t_d+\tau_{\mathrm{upper}},
$$

where the positive tolerances are configured in seconds through `timeToleranceLower` and `timeToleranceUpper`. A zero tolerance disables its corresponding bound independently. The current settings, `timeToleranceLower=1.0` and `timeToleranceUpper=0.0`, allow acquisition from one second before $t_d$ onward, without an instrument upper timing bound. A late acquisition is therefore possible if the pointing requirements are met later. Setting both tolerances to zero removes both timing gates, so the requested time no longer delays the instrument.

Physical access, action-duration limits, and simulation stepping still apply. A positive upper tolerance also lets the native imaging action limit its timeout using the requested delay plus that tolerance. With the upper tolerance disabled, this example retains the physical window and existing action/environment limits; the fraction does not impose a new upper deadline.


In [7]:
from bsk_rl.gym import is_no_action


def clean_action(
    action: tuple[int, float | np.ndarray | list],
) -> tuple[int, float]:
    if action is None:
        return None
    discrete = action[0].item() if isinstance(action[0], np.generic) else action[0]
    val = action[1][0] if isinstance(action[1], (list, np.ndarray)) else action[1]
    continuous = val.item() if isinstance(val, np.generic) else val

    return (discrete, continuous)


class WrapperActionDuration(gym.ActionWrapper):
    """Convert a fraction into an acquisition delay over the remaining window.

    The returned delay is in seconds and has a minimum of 0.5 seconds.
    """

    def __init__(
        self,
        env,
    ) -> None:
        super().__init__(env)
        self.env = env

        self._n_ahead = {}
        for sat in self.env.satellites:
            obs_class = next(
                item
                for item in sat.observation_spec
                if isinstance(item, (obs.OpportunityProperties))
                and item.type == "target"
            )
            self._n_ahead[sat.name] = obs_class.n_ahead_observe

    def get_image_window(
        self, target_index: int, sat_name: str | None
    ) -> tuple[float, float]:

        if sat_name is not None:
            sat = next(sat for sat in self.unwrapped.satellites if sat.name == sat_name)
        else:
            sat = self.unwrapped.satellite

        opps = sat.find_next_opportunities(
            n=self._n_ahead[sat.name],
            types="target",
            pad=True,
        )

        if isinstance(target_index, (int, np.integer)):
            opp = opps[target_index]
        else:
            opp = [
                tgt_i
                for tgt_i in opps
                if not isinstance(tgt_i["object"], str)
                and tgt_i["object"].id == target_index
            ]
            if len(opp) > 0:
                opp = opp[0]
            else:
                for i in range(5):
                    opps = sat.find_next_opportunities(
                        n=self._n_ahead[sat.name] * (i + 1),
                        types="target",
                        pad=True,
                    )
                    opp = [
                        tgt_i
                        for tgt_i in opps
                        if not isinstance(tgt_i["object"], str)
                        and tgt_i["object"].id == target_index
                    ]
                    if len(opp) > 0:
                        opp = opp[0]
                        break

        window_start = opp["window"][0] - self.unwrapped.simulator.sim_time
        window_start = np.max([window_start, 0.0])
        window_end = opp["window"][1] - self.unwrapped.simulator.sim_time

        return (window_start, window_end)

    def get_imaging_time(
        self, target_index: int, window_fraction: float, sat_name: str | None
    ) -> float:
        window_start, window_end = self.get_image_window(target_index, sat_name)
        window_fraction = np.clip(window_fraction, 0.0, 1.0)
        return np.clip(
            float(window_start + (window_end - window_start) * window_fraction),
            0.5,
            np.inf,
        )

    def _set_action_duration(
        self, action, sat_name: str | None
    ) -> str | None | tuple[int, float]:
        if isinstance(action, str) or action is None:
            return action

        action = clean_action(action)
        target_index = action[0]
        duration = self.get_imaging_time(target_index, action[1], sat_name)

        return (action[0], float(duration))

    def action(
        self, action: tuple[int, float]
    ) -> tuple[int, float] | dict | str | None:

        if isinstance(action, dict):
            action_dict = {}
            for sat_name, sat_action in action.items():
                action_dict[sat_name] = self._set_action_duration(sat_action, sat_name)
            return action_dict

        else:
            return self._set_action_duration(action, None)


class WrapperDiscreteToHybridAction(gym.ActionWrapper, gym.Wrapper):
    """Decode a categorical choice into a target index and imaging fraction.

    Timing options vary fastest within each target: action = target * D + timing.

    Args:
        env: Imaging environment with a native hybrid action space.
        discrete_imaging_fraction: Predefined timing fractions for each target.
    """

    def __init__(
        self,
        env,
        discrete_imaging_fraction: Iterable[float],
    ) -> None:
        super().__init__(env)
        self.discrete_imaging_fraction = list(discrete_imaging_fraction)
        self.discretization = len(self.discrete_imaging_fraction)
        self.env = env
        if not isinstance(env.action_space, gym.spaces.Tuple):
            action_space = env.action_spaces
            # Assumes all satellites have the same action space
            self.n_hybrid_actions = action_space[env.satellites[0].name][0].n
        else:
            action_space = env.action_space[0]
            self.n_hybrid_actions = action_space.n
        self.n_discrete_actions = self.n_hybrid_actions * self.discretization
        space = gym.spaces.Discrete(self.n_discrete_actions)
        if isinstance(env.action_space, gym.spaces.Tuple):
            self.action_space = space
        else:
            self.action_space = lambda sat_name: space

    def _split_action(self, action):
        if action is None or is_no_action(action):
            return None

        action = int(action)
        if not 0 <= action < self.n_discrete_actions:
            raise ValueError(
                f"Action index {action} out of range for {self.n_discrete_actions} choices."
            )
        target_index, continuous_action = divmod(action, self.discretization)
        return (target_index, self.discrete_imaging_fraction[continuous_action])

    def action(
        self, action: int | dict
    ) -> (
        tuple[int, float]
        | tuple[str, float]
        | dict[str, tuple[int, float]]
        | dict[str, tuple[str, float]]
        | None
    ):
        if isinstance(action, dict):
            action_dict = {}
            for sat_name, sat_action in action.items():
                action_dict[sat_name] = self._split_action(sat_action)
            return action_dict
        else:
            return self._split_action(action)


class WrapperActionLogging(gym.Wrapper):
    def __init__(self, env):

        super().__init__(env)
        self.env = env

        if not isinstance(env.action_space, gym.spaces.Tuple):
            action_space = env.action_spaces
            self.n_discrete = action_space[env.satellites[0].name][0].n
        else:
            action_space = env.action_space
            self.n_discrete = action_space[0].n

        self._initialize_action_logger()

    def _initialize_action_logger(self):

        self.action_duration_logger_list = {
            sat.name: {
                "duration_image": [],
            }
            for sat in self.env.satellites
        }

        self.action_log_discrete_count = {i: 0 for i in range(self.n_discrete)}
        self.action_duration_log = {i: 0.0 for i in range(self.n_discrete)}

    def return_loggers(self):

        duration_list_image_all = [
            self.action_duration_logger_list[sat.name]["duration_image"]
            for sat in self.env.satellites
        ]

        return {
            "action_log_discrete_count": self.action_log_discrete_count,
            "action_duration_log": self.action_duration_log,
            "action_duration_logger_list": self.action_duration_logger_list,
            "action_duration_image_list_all": np.concatenate(duration_list_image_all),
        }

    def reset(self, *, seed: int | None = None, options: dict[str, Any] | None = None):
        self._initialize_action_logger()
        return self.env.reset(seed=seed, options=options)

    def _store_action_info(self, action: tuple[int, float], sat_name: str):

        action = clean_action(action)
        if action is None:
            return

        self.action_duration_logger_list[sat_name]["duration_image"].append(action[1])

        if isinstance(action[0], str):
            return
        discrete_action = int(action[0])
        self.action_log_discrete_count[discrete_action] += 1
        self.action_duration_log[discrete_action] += (
            action[1] - self.action_duration_log[discrete_action]
        ) / self.action_log_discrete_count[discrete_action]

    def step(self, action: dict | tuple[int, float]):

        if isinstance(action, dict):
            for sat_name, sat_action in action.items():
                self._store_action_info(sat_action, sat_name)
        else:
            self._store_action_info(action, self.env.satellite.name)

        return self.env.step(action)


ENV_CONFIG = dict(
    n_targets=(100, 3000),
    orbits=3,
    quality_weight=(0.0, 1.0),  # Scalar for a common weight; 0.0 disables quality.
    quality_types=("quality_off_nadir_angle", "quality_periodic"),
)


def reward_function(priority, quality, quality_weight=0.5):

    quality_weight = np.clip(quality_weight, 0.0, 1.0)

    quality = (1 - quality_weight) + quality_weight * np.clip(
        quality, a_min=0.0, a_max=1.0
    )
    reward = priority * quality

    return reward

### Create consistent training and evaluation environments

`make_env` combines the target scenario, rewarder, and simulation settings from a common configuration. In evaluation mode it returns a `ConstellationTasking` environment with the requested number of satellites, including a per-satellite dictionary interface when only one satellite is used. With `train=True`, it returns the construction arguments for a single-satellite `SatelliteTasking` environment; the RLlib wrapper below uses these arguments to instantiate the training environment.

`setup_wrapper_stack` then adds duration logging and converts a normalized imaging fraction into a delay in seconds. Setting `discrete_action_space=True` also adds the outer wrapper that decodes an integer into a target and one of the supplied timing fractions. Keeping environment construction and action conversion in these helpers makes the settings and action convention explicit when creating a training or evaluation instance.


In [8]:
def make_env(
    env_config=None,
    n_sats=1,
    incremental_score=False,
    n_targets: int | None = None,
    train: bool = False,
    time_limit: float | None = None,
):
    config = {**ENV_CONFIG, **dict(env_config or {})}
    env_config = dict(
        scenario=QualityTargets(
            n_targets=config["n_targets"] if n_targets is None else n_targets,
            quality_weights=config["quality_weight"],
            quality_function_name=config["quality_types"],
        ),
        rewarder=QualityImageReward(
            reward_fn=reward_function, incremental_score=incremental_score
        ),
        sim_rate=SIM_RATE,
        max_step_duration=500.0,
        time_limit=config["orbits"] * ORBIT_PERIOD
        if time_limit is None
        else time_limit,
        failure_penalty=0.0,
        log_level="INFO",
    )
    if train:
        # Training uses a single satellite configuration
        env_config.update(
            dict(satellite=QualitySatellite("EO", sat_args=satellite_args()))
        )
        return env_config
    else:
        env = ConstellationTasking(
            satellites=[
                QualitySatellite(f"EO_{i}", sat_args=satellite_args())
                for i in range(n_sats)
            ],
            **env_config,
        )
        return env


def setup_wrapper_stack(
    env,
    discrete_action_space=False,
    discrete_imaging_fraction=None,
):

    env = WrapperActionLogging(env)
    env = WrapperActionDuration(env)
    if discrete_action_space:
        env = WrapperDiscreteToHybridAction(
            env,
            discrete_imaging_fraction=discrete_imaging_fraction,
        )

    return env

## Try a few imaging actions

This short rollout selects target opportunity 0, the first upcoming opportunity, and requests the midpoint of its remaining access window with `(0, 0.5)`. It uses `discrete_action_space=False`, so the action contains a target index and a timing fraction directly. The action is supplied in a dictionary keyed by satellite name, and the duration wrapper converts the fraction into seconds. This exercises the physical environment without RLlib or a trained checkpoint. Increase the loop length to run a longer rollout.


In [9]:
env = make_env()

env = setup_wrapper_stack(
    env,
    discrete_action_space=False,
    discrete_imaging_fraction=IMAGING_FRACTIONS,
)

observation, info = env.reset(seed=3)
try:
    for _ in range(5):
        # Each satellite receives (target index, window fraction).
        # With the discrete wrapper, each dictionary value is an action index.
        action = {env.satellites[0].name: (0, 0.5)}
        observation, reward, terminated, truncated, info = env.step(action)
        if all(terminated.values()) or all(truncated.values()):
            break
finally:
    env.close()

2026-10-26 21:02:03,844 gym                            INFO       Resetting environment with seed=3
2026-10-26 21:02:03,846 scene.targets                  INFO       Generating 1742 targets
2026-10-26 21:02:04,129 sats.satellite.EO_0            INFO       <0.00> EO_0: Finding opportunity windows from 0.00 to 18600.00 seconds
2026-10-26 21:02:04,419 gym                            INFO       <0.00> Environment reset
2026-10-26 21:02:04,421 gym                            INFO       <0.00> === STARTING STEP ===
2026-10-26 21:02:04,422 sats.satellite.EO_0            INFO       <0.00> EO_0: target index 0 tasked
2026-10-26 21:02:04,422 sats.satellite.EO_0            INFO       <0.00> EO_0: Target(Villa Aberastain, AR) imaging timing: requested capture at 417.2
2026-10-26 21:02:04,423 sats.satellite.EO_0            INFO       <0.00> EO_0: Target(Villa Aberastain, AR) tasked for imaging
2026-10-26 21:02:04,424 sats.satellite.EO_0            INFO       <0.00> EO_0: Target(Villa Aberastain, AR) 

## Categorical policy over target/timing pairs

Because timing choices are discrete, the policy only needs RLlib's standard `TorchCategorical`. The actor produces one logit per target and timing option, then flattens them into the same target-major order as the wrapper. A single softmax represents the joint distribution $\pi(i,k\mid o)$, so timing preferences can depend on the target without a custom conditional distribution class.

A `MultiDiscrete([N, D])` policy with separate categorical factors would impose independence between target and timing choices. The flattened categorical avoids that restriction. A custom distribution would become useful if capture times were continuous and conditioned on the chosen target.


In [10]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.algorithms.ppo.torch.ppo_torch_rl_module import PPOTorchRLModule
from ray.rllib.core import Columns
from ray.rllib.core.rl_module.rl_module import RLModuleSpec
from ray.rllib.models.torch.torch_distributions import TorchCategorical
from ray.rllib.utils.framework import try_import_torch

from bsk_rl.utils.rllib.discounting import TimeDiscountedGAEPPOTorchLearner

torch, nn = try_import_torch()

2026-10-26 21:02:05,406	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-10-26 21:02:07,388	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## Configurable actor, critic, and combined RL module

The following definitions construct a custom actor, which outputs $ND$ logits, and a custom critic:

* **Actor:** separate target and satellite MLPs; multi-head target self-attention with a residual connection and LayerNorm; mean/max target pooling; and one logit per target/timing pair.
* **Critic:** a separate target MLP, mean/max pooling, and an MLP combining these embeddings with the raw satellite/context features to predict a scalar value.

Both networks accept keyword arguments for their dimensions, depths, activation, and dropout. `HybridQualityModule` combines them and reads those arguments from
RLlib's `model_config_dict`; critic arguments use the `critic_` prefix. The module checks that its target count and timing discretization agree with the environment. The training forward pass returns logits and values so PPO can evaluate the actions stored in its rollout batch.

In [11]:
def build_mlp(
    input_size,
    width,
    depth,
    output_size,
    *,
    act_function="ReLU",
    dropout=0.0,
    dropout_before_output=0.0,
    output_activation=False,
):
    """Configurable hidden layers, preserving the research parameter layout."""
    if depth < 0 or width <= 0 or output_size <= 0:
        raise ValueError("MLP dimensions must be positive and depth nonnegative.")
    activation = getattr(nn, act_function)
    layers = []
    for _ in range(depth):
        layers.extend([nn.Linear(input_size, width), activation()])
        if dropout:
            layers.append(nn.Dropout(dropout))
        input_size = width
    if dropout_before_output:
        layers.append(nn.Dropout(dropout_before_output))
    layers.append(nn.Linear(input_size, output_size))
    if output_activation:
        layers.append(activation())
    return nn.Sequential(*layers)


class TargetSelfAttention(nn.Module):
    def __init__(self, input_dim, attention_dim, num_heads):
        super().__init__()
        if num_heads <= 0 or attention_dim <= 0 or attention_dim % num_heads:
            raise ValueError(
                "attention_dim must be positive and divisible by num_heads."
            )
        self.num_heads = num_heads
        self.head_dim = attention_dim // num_heads
        self.attention_dim = attention_dim
        self.qkv = nn.Linear(input_dim, 3 * attention_dim, bias=False)
        self.out = nn.Linear(attention_dim, input_dim, bias=False)

    def forward(self, tokens):
        batch_size, n_targets, _ = tokens.shape
        qkv = self.qkv(tokens).view(
            batch_size, n_targets, 3, self.num_heads, self.head_dim
        )
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)
        attended = nn.functional.scaled_dot_product_attention(
            q, k, v, dropout_p=0.0, is_causal=False
        )
        attended = (
            attended.transpose(1, 2)
            .contiguous()
            .view(batch_size, n_targets, self.attention_dim)
        )
        return self.out(attended)


class ImagingActor(nn.Module):
    def __init__(
        self,
        target_size,
        context_size,
        n_times,
        *,
        width_phi=256,
        depth_phi=2,
        tgt_encoded_dim=64,
        attention_dim=64,
        num_heads=2,
        width_phi_sat=256,
        depth_phi_sat=2,
        sat_encoded_dim=64,
        width_psi_img=128,
        depth_psi_img=4,
        act_function="ReLU",
        dropout=0.0,
        psi_img_dropout_last_layer=0.1,
    ):
        super().__init__()
        self.phi = build_mlp(
            target_size,
            width_phi,
            depth_phi,
            tgt_encoded_dim,
            act_function=act_function,
            dropout=dropout,
        )
        self.self_attention = TargetSelfAttention(
            tgt_encoded_dim, attention_dim, num_heads
        )
        self.norm_self_attention = nn.LayerNorm(tgt_encoded_dim)
        self.psi_img = build_mlp(
            3 * tgt_encoded_dim + sat_encoded_dim,
            width_psi_img,
            depth_psi_img,
            tgt_encoded_dim,
            act_function=act_function,
            dropout=dropout,
            dropout_before_output=psi_img_dropout_last_layer,
            output_activation=True,
        )
        self.out_layer_psi_img = nn.Linear(tgt_encoded_dim, n_times)
        self.phi_sat = build_mlp(
            context_size,
            width_phi_sat,
            depth_phi_sat,
            sat_encoded_dim,
            act_function=act_function,
            dropout=dropout,
        )

    def forward(self, context, targets):
        tokens = self.phi(targets)
        satellite = self.phi_sat(context)
        tokens = self.norm_self_attention(tokens + self.self_attention(tokens))
        n_targets = tokens.shape[1]
        mean_targets = tokens.mean(dim=1, keepdim=True).expand(-1, n_targets, -1)
        max_targets = tokens.max(dim=1, keepdim=True).values.expand(-1, n_targets, -1)
        features = torch.cat(
            [
                tokens,
                mean_targets,
                max_targets,
                satellite.unsqueeze(1).expand(-1, n_targets, -1),
            ],
            dim=-1,
        )
        logits = self.out_layer_psi_img(self.psi_img(features))
        return logits.flatten(start_dim=1)


class ImagingCritic(nn.Module):
    def __init__(
        self,
        target_size,
        context_size,
        *,
        width_phi=256,
        depth_phi=2,
        tgt_encoded_dim=128,
        width_psi=64,
        depth_psi=3,
        act_function="ReLU",
        dropout=0.0,
    ):
        super().__init__()
        self.phi = build_mlp(
            target_size,
            width_phi,
            depth_phi,
            tgt_encoded_dim,
            act_function=act_function,
            dropout=dropout,
        )
        self.psi = build_mlp(
            context_size + 2 * tgt_encoded_dim,
            width_psi,
            depth_psi,
            1,
            act_function=act_function,
            dropout=dropout,
        )

    def forward(self, context, targets):
        tokens = self.phi(targets)
        features = torch.cat(
            [context, tokens.mean(dim=1), tokens.max(dim=1).values], dim=-1
        )
        return self.psi(features).squeeze(-1)


class HybridQualityModule(PPOTorchRLModule):
    """Combine independently configurable actor and critic heads for PPO."""

    def setup(self):
        config = self.config.model_config_dict
        self.n_targets = int(config["n_targets"])
        self.context_size = int(config["obs_sat"])
        self.n_times = int(config["discretization"])
        target_features = self.config.observation_space.shape[0] - self.context_size
        if (
            self.n_targets <= 0
            or self.context_size < 0
            or target_features <= 0
            or target_features % self.n_targets
        ):
            raise ValueError(
                "Observation size must match context plus equal target tokens."
            )
        self.target_size = target_features // self.n_targets
        if (
            not isinstance(self.config.action_space, gym.spaces.Discrete)
            or self.n_times <= 0
            or self.config.action_space.n != self.n_targets * self.n_times
        ):
            raise ValueError("Expected Discrete(n_targets * discretization) actions.")
        actor_keys = (
            "width_phi",
            "depth_phi",
            "tgt_encoded_dim",
            "attention_dim",
            "num_heads",
            "width_phi_sat",
            "depth_phi_sat",
            "sat_encoded_dim",
            "width_psi_img",
            "depth_psi_img",
            "act_function",
            "dropout",
            "psi_img_dropout_last_layer",
        )
        self.pi_head = ImagingActor(
            self.target_size,
            self.context_size,
            self.n_times,
            **{key: config[key] for key in actor_keys if key in config},
        )
        self.vf = ImagingCritic(
            self.target_size,
            self.context_size,
            width_phi=config.get("critic_width_phi", 256),
            depth_phi=config.get("critic_depth_phi", 2),
            tgt_encoded_dim=config.get("critic_tgt_encoded_dim", 128),
            width_psi=config.get("critic_width_psi", 64),
            depth_psi=config.get("critic_depth_psi", 3),
            act_function=config.get("act_function", "ReLU"),
            dropout=config.get("critic_dropout", config.get("dropout", 0.0)),
        )
        self.action_dist_cls = TorchCategorical
        self._inference_only_state_dict_keys = {}

    def get_initial_state(self):
        return {}

    def split_observation(self, batch):
        values = batch[Columns.OBS].float()
        context = values[:, : self.context_size]
        targets = values[:, self.context_size :].reshape(
            -1, self.n_targets, self.target_size
        )
        return context, targets

    def compute_values(self, batch, embeddings=None):
        return self.vf(*self.split_observation(batch))

    def _forward_inference(self, batch, **kwargs):
        return {
            Columns.ACTION_DIST_INPUTS: self.pi_head(*self.split_observation(batch))
        }

    def _forward_exploration(self, batch, **kwargs):
        return self._forward_train(batch)

    def _forward_train(self, batch, **kwargs):
        context, targets = self.split_observation(batch)
        return {
            Columns.ACTION_DIST_INPUTS: self.pi_head(context, targets),
            Columns.VF_PREDS: self.vf(context, targets),
        }

## PPO configuration

The configuration below supplies the model dimensions and PPO hyperparameters for three timing options per target. `Env_wrapped` adapts the custom environment for RLlib: it accepts `env_config`, constructs the single-satellite environment, applies the action-wrapper stack, and initializes the optional episode-data logging callbacks. Passing the class directly as `env=Env_wrapped` lets RLlib create each environment instance without registering a separate Gym environment ID.

`TimeDiscountedGAEPPOTorchLearner` uses the reported step duration `d_ts`, in seconds, to apply the duration-dependent discount $\gamma^{\Delta t}$. Here `gamma=0.995` is the per-second factor, and the learner discounts raw capture rewards at the end of each step. The evaluation loop accumulates undiscounted rewards, as in the paper's reported comparisons.

Defining `ppo_config` does not build or train an algorithm. The inference demonstration at the end of the cell creates a separate wrapped environment and a fresh module with random initial weights, then executes one categorical action. Its output checks the inference interface and does not represent trained-policy performance.


In [12]:
from bsk_rl.utils.rllib.callbacks import EpisodeDataLogger

MODEL_CONFIG = {
    "n_targets": N_AHEAD,
    "obs_sat": 32,
    "discretization": len(IMAGING_FRACTIONS),
    "width_phi": 256,
    "depth_phi": 2,
    "tgt_encoded_dim": 64,
    "attention_dim": 64,
    "num_heads": 2,
    "width_phi_sat": 256,
    "depth_phi_sat": 2,
    "sat_encoded_dim": 64,
    "width_psi_img": 128,
    "depth_psi_img": 4,
    "act_function": "ReLU",
    "dropout": 0.0,
    "psi_img_dropout_last_layer": 0.1,
    "critic_tgt_encoded_dim": 128,
    "critic_width_phi": 256,
    "critic_depth_phi": 2,
    "critic_width_psi": 64,
    "critic_depth_psi": 3,
}
TRAINING_CONFIG = {
    "gamma": 0.995,
    "lr": [[0, 0.00012122027544761331], [117126, 3.707168645103191e-06]],
    "entropy_coeff": [[0, 0.047466805372264546], [734487, 0.0]],
    "train_batch_size": 9000,
    "num_sgd_iter": 15,
    "lambda_": 0.8713548569911232,
    "use_kl_loss": False,
    "clip_param": 0.24943882616708635,
    "grad_clip": 0.3104924935285628,
    "vf_clip_param": 2.0,
}


module_spec = RLModuleSpec(
    module_class=HybridQualityModule, model_config_dict=dict(MODEL_CONFIG)
)


class Env_wrapped(EpisodeDataLogger, gym.Wrapper):
    def __init__(self, env_config):

        env_config = dict(env_config)
        episode_data_callback = env_config.pop("episode_data_callback", None)
        satellite_data_callback = env_config.pop("satellite_data_callback", None)
        env_args = make_env(env_config, n_sats=1, train=True)
        env = SatelliteTasking(**env_args)
        env = WrapperActionLogging(
            env,
        )
        env = WrapperActionDuration(
            env,
        )
        env = WrapperDiscreteToHybridAction(
            env,
            discrete_imaging_fraction=IMAGING_FRACTIONS,
        )
        EpisodeDataLogger.__init__(self, episode_data_callback, satellite_data_callback)
        gym.Wrapper.__init__(self, env)


ppo_config = (
    PPOConfig()
    .environment(
        env=Env_wrapped,
        env_config=dict(ENV_CONFIG),
        normalize_actions=False,
        clip_actions=False,
    )
    .framework("torch")
    .api_stack(
        enable_rl_module_and_learner=True,
        enable_env_runner_and_connector_v2=True,
    )
    .env_runners(num_env_runners=0, rollout_fragment_length=32)
    .training(learner_class=TimeDiscountedGAEPPOTorchLearner, **TRAINING_CONFIG)
    .rl_module(rl_module_spec=module_spec)
)

env = make_env()

env = setup_wrapper_stack(
    env,
    discrete_action_space=True,
    discrete_imaging_fraction=IMAGING_FRACTIONS,
)

env.reset(seed=7)
module = RLModuleSpec(
    module_class=HybridQualityModule,
    observation_space=env.observation_space(env.satellites[0].name),
    action_space=env.action_space(env.satellites[0].name),
    model_config_dict=dict(MODEL_CONFIG),
).build()


def policy(satellite, *args, **kwargs):
    obs = torch.as_tensor(satellite.get_obs())

    module.eval()
    with torch.no_grad():
        output = module.forward_inference({Columns.OBS: obs[None]})
        distribution = module.get_inference_action_dist_cls().from_logits(
            output[Columns.ACTION_DIST_INPUTS]
        )
        action = distribution.to_deterministic().sample()

    return int(action.item())


action = policy(env.satellites[0])

_, reward, _, _, info = env.step({env.satellites[0].name: action})
target_index, timing_index = divmod(action, len(IMAGING_FRACTIONS))
print(
    "Untrained policy:",
    action,
    "Target:",
    target_index,
    "Imaging fraction:",
    IMAGING_FRACTIONS[timing_index],
    "Reward:",
    reward,
)

2026-10-26 21:02:08,848                                WARNING    Creating logger for new env on PID=71679. Old environments in process may now log times incorrectly.
2026-10-26 21:02:08,851 gym                            INFO       Resetting environment with seed=7
2026-10-26 21:02:08,855 scene.targets                  INFO       Generating 390 targets
2026-10-26 21:02:08,986 sats.satellite.EO_0            INFO       <0.00> EO_0: Finding opportunity windows from 0.00 to 18600.00 seconds
2026-10-26 21:02:09,092 gym                            INFO       <0.00> Environment reset
2026-10-26 21:02:09,141 gym                            INFO       <0.00> === STARTING STEP ===
2026-10-26 21:02:09,142 sats.satellite.EO_0            INFO       <0.00> EO_0: target index 38 tasked
2026-10-26 21:02:09,143 sats.satellite.EO_0            INFO       <0.00> EO_0: Target(Bougara, DZ) imaging timing: requested capture at 3638.7
2026-10-26 21:02:09,144 sats.satellite.EO_0            INFO       <0.00> EO_

Untrained policy: 115 Target: 38 Imaging fraction: 0.5 Reward: {'EO_0': 0.0}


## Multi-satellite coordination through communication

A policy trained for one satellite can be evaluated independently on each satellite, with coordination handled by the decision loop below. This example models ideal, instantaneous sharing of completed acquisitions and intended tasks. It does not simulate message transmission delays or communication outages. The `policy` function below uses deterministic inference from the **untrained module** created above; load trained weights to evaluate a learned RL policy.

An **intent** identifies a target another satellite is currently tasked to image. Reward-aware intents also carry the **expected absolute reward** at the requested acquisition time. This prediction is not an obtained reward and does not guarantee capture at the requested time.

The human-readable mode names correspond to these `multisat_comm_config["mode"]` values:

| Mode | Code value | Coordination rule |
|---|---|---|
| **Intent** | `"intent"` | Exclude globally imaged targets and targets currently intended by other satellites. A new decision selects from the remaining targets; there is no comparison of competing reward predictions. |
| **Intent-incremental** | `"incremental_intent"` | Keep re-imaging candidates when their predicted reward can exceed both the best completed reward and the largest relevant competing intent reward. Store this maximum in bookkeeping and use the clipped marginal profile to test whether an opportunity has a useful improvement. Several satellites may retain intents for the same target; this mode does not forcibly choose a single claimant. |
| **Intent-greedy** | `"greedy"` | Exclude globally imaged targets. If a satellite selects an already intended target, compare the two expected **absolute** rewards at their respective requested acquisition times. A strictly larger prediction replaces the previous claimant and schedules that satellite to choose again. Otherwise, temporarily exclude the contested target for the challenger and ask it to choose again. Equal predictions retain the existing claimant. |
| **Intent-greedy-incremental** | `"greedy_incremental"` | Use the same conflict-resolution rule as intent-greedy while allowing useful re-imaging. The marginal profile is measured against the best **completed** reward; pending intents are handled by the explicit greedy comparison rather than subtracted from this baseline. |

The additional `"imaged"` mode shares completed acquisitions only: it excludes globally imaged targets without reserving targets through intents. Nonincremental modes are intended to use `incremental_score=False`; the two incremental modes require `incremental_score=True`. Select rewarder scoring explicitly when creating the environment—the communication-mode string does not change it automatically.

For a given target and the satellite currently choosing an action, intent-incremental computes:

```python
baseline = max(best_obtained_reward, highest_competing_intent_reward)
marginal_reward = max(absolute_reward_at_t - baseline, 0)
```

With no competing intent, the baseline is simply the best obtained reward. Intent-greedy-incremental uses that completed-reward baseline and resolves competing intents separately. Advertising an intent does not increase obtained reward. Actual incremental reward is the improvement over the best completed reward, not over an intent prediction.

Before each decision, the loop refreshes the satellite's completed/intended bookkeeping and clears the cached observation. Intent-based exclusion changes which target opportunities are available. Incremental modes additionally use `estimate_quality_profile(..., return_rewards=True)` to retain opportunities whose maximum predicted marginal reward exceeds `0.001`. This is an opportunity-level check; it does not guarantee a positive reward at the policy's chosen timing option. `cooldown_s` is a separate intended restriction on rapid captures of the same target by the same satellite.

**Observation wiring:** `QualitySatellite` binds its spline observation to `estimate_quality_profile`. After the loop updates bookkeeping and clears the observation cache, the next observation fits the marginal profile using that satellite's current completed/intended reward baseline. The estimator caches absolute predictions but subtracts the current bookkeeping baseline on every call, so same-time bookkeeping changes affect the spline coefficients. This changes observation values while preserving the 512-value observation shape. The successful-capture handler also records `last_img_time_per_sat`, which supplies the cooldown filter.


### Helpers for decoding and comparing imaging actions

`get_imaging_action` decodes an action into a target index and normalized timing fraction, or returns `None` when no imaging action is supplied. `get_time` uses the duration wrapper to convert that fraction into a delay in seconds for the selected satellite and target opportunity.

`get_expected_absolute_reward` adds the current simulation time to this delay and evaluates the predicted absolute reward at the requested acquisition time. The coordination loop uses this prediction to advertise reward-aware intents and compare competing tasks. It is evaluated independently of the bookkeeping baseline: marginal reward is computed separately from completed or intended rewards. The requested time remains a prediction point; it does not guarantee that the instrument will capture the image at that time.


In [13]:
def get_expected_absolute_reward(env, satellite, target_index, action_fraction):
    """Predict absolute reward at the wrapper-decoded desired acquisition time.

    The wrapper returns a delay, whereas quality functions expect absolute
    simulator time. The prediction is independent of reward bookkeeping and
    does not guarantee acquisition at the desired time or within the window.
    """
    target = satellite.parse_target_selection(target_index)
    imaging_delay = float(
        env.get_imaging_time(target_index, action_fraction, satellite.name)
    )
    if not np.isfinite(imaging_delay) or imaging_delay < 0.0:
        raise ValueError(f"Invalid desired imaging delay: {imaging_delay}")
    desired_time = satellite.simulator.sim_time + imaging_delay
    predicted_reward = estimate_absolute_reward(
        satellite, target, np.array([desired_time], dtype=float)
    )
    if predicted_reward.size != 1 or not np.all(np.isfinite(predicted_reward)):
        raise ValueError(
            f"Invalid expected acquisition reward for target {target.id}: "
            f"{predicted_reward}"
        )
    return float(predicted_reward.item())


def get_imaging_action(env, action):
    """Decode a target index, or return None for a non-imaging action."""
    action_parsed = (
        env._split_action(action)
        if isinstance(env, WrapperDiscreteToHybridAction)
        else action
    )
    if action_parsed is None:
        return None
    tgt_index, fraction = action_parsed
    return tgt_index, fraction


def get_time(env, action: int | tuple[int, float], sat_name: str) -> float:

    action_parsed = get_imaging_action(env, action)
    if action_parsed is None:
        raise ValueError("Non-imaging actions do not have an imaging time.")

    action_time = env.get_imaging_time(action_parsed[0], action_parsed[1], sat_name)

    return action_time

### Configure satellite access filters

The following block creates the constellation and configures each satellite's access filters for the selected communication mode. These filters determine which target opportunities are available when forming observations and selecting actions. Depending on the mode, they exclude completed or competing intended targets, temporarily exclude targets lost in greedy comparisons, or retain opportunities with useful predicted marginal reward.

The displayed configuration uses two satellites, 1000 targets, and `incremental_intent` with incremental scoring enabled. Its filters require a maximum predicted marginal reward greater than `0.001` over the remaining window and more than `cooldown_s=15.0` seconds since the same satellite last captured that target. These checks select eligible opportunities; they do not guarantee positive reward at the chosen timing fraction. The 15-second cooldown is an explicit setting of this example; the current paper draft does not specify a cooldown value.


In [ ]:
from copy import copy

setup_env_args = {}
incremental_score = True
cooldown_s = 15.0  # set to 0 to disable cooldown
seed = 42
n_targets = 1000
n_sats = 2
multisat_comm_config = {"mode": "incremental_intent"}

env = make_env(
    setup_env_args,
    n_sats=n_sats,
    incremental_score=incremental_score,
    n_targets=n_targets,
    time_limit=ORBIT_PERIOD * 0.1,
)

env = setup_wrapper_stack(
    env, discrete_action_space=True, discrete_imaging_fraction=IMAGING_FRACTIONS
)

seed_case = seed + n_targets
_, info = env.reset(seed=seed_case)

sat_last_decision_time = {sat.name: 0.0 for sat in env.satellites}
reward_cumulative = 0.0

intents = {sat.name: None for sat in env.satellites}
intents_quality = {sat.name: None for sat in env.satellites}

comm_mode = multisat_comm_config["mode"]
if len(env.satellites) > 1:

    def filter_intent(opp, sat):
        return opp["object"] not in sat.intent_set_shared

    def filter_imaged(opp, sat):
        return opp["object"] not in sat.imaged_set_shared

    def filter_minimum_reward(opp, sat):
        rewards = estimate_quality_profile(sat, opp, return_rewards=True)
        minimum_quality = np.max(rewards) > 1e-3
        return minimum_quality

    def filter_time(opp, sat):
        tgt = opp["object"]
        current_time = sat.simulator.sim_time
        time_diff = current_time - tgt.last_img_time_per_sat.get(sat.name, -np.inf)
        return time_diff > cooldown_s

    def filter_lower_reward(opp, sat):
        return opp["object"] not in sat.temp_filter_set

    for sat in env.satellites:
        sat.intent_set_shared = set()
        sat.imaged_set_shared = set()

    if comm_mode == "imaged":
        for sat in env.satellites:
            sat.add_access_filter(lambda opp, sat=sat: filter_imaged(opp, sat))

    elif comm_mode == "intent":
        for sat in env.satellites:
            sat.add_access_filter(lambda opp, sat=sat: filter_intent(opp, sat))
            sat.add_access_filter(lambda opp, sat=sat: filter_imaged(opp, sat))

    elif comm_mode == "incremental_intent":
        for sat in env.satellites:
            if multisat_comm_config.get("filter_minimum_quality", True):
                sat.add_access_filter(
                    lambda opp, sat=sat: filter_minimum_reward(opp, sat)
                )
            if cooldown_s > 0:
                sat.add_access_filter(lambda opp, sat=sat: filter_time(opp, sat))

    elif comm_mode == "greedy":
        # Retask an existing claimant when another satellite predicts higher reward.

        for sat in env.satellites:
            sat.temp_filter_set = set()
            sat.add_access_filter(lambda opp, sat=sat: filter_imaged(opp, sat))
            sat.add_access_filter(lambda opp, sat=sat: filter_lower_reward(opp, sat))

    elif comm_mode == "greedy_incremental":
        for sat in env.satellites:
            sat.temp_filter_set = set()
            sat.add_access_filter(lambda opp, sat=sat: filter_lower_reward(opp, sat))
            if multisat_comm_config.get("filter_minimum_quality", True):
                sat.add_access_filter(
                    lambda opp, sat=sat: filter_minimum_reward(opp, sat)
                )
            if cooldown_s > 0:
                sat.add_access_filter(lambda opp, sat=sat: filter_time(opp, sat))

2026-10-26 21:02:09,304                                WARNING    Creating logger for new env on PID=71679. Old environments in process may now log times incorrectly.
2026-10-26 21:02:09,307 gym                            INFO       Resetting environment with seed=1042
2026-10-26 21:02:09,311 scene.targets                  INFO       Generating 1000 targets
2026-10-26 21:02:09,499 sats.satellite.EO_0            INFO       <0.00> EO_0: Finding opportunity windows from 0.00 to 1800.00 seconds
2026-10-26 21:02:09,561 sats.satellite.EO_1            INFO       <0.00> EO_1: Finding opportunity windows from 0.00 to 1800.00 seconds
2026-10-26 21:02:09,647 sats.satellite.EO_0            INFO       <0.00> EO_0: Finding opportunity windows from 1800.00 to 4200.00 seconds
2026-10-26 21:02:09,703 sats.satellite.EO_1            INFO       <0.00> EO_1: Finding opportunity windows from 1800.00 to 4200.00 seconds
2026-10-26 21:02:09,837 gym                            INFO       <0.00> Environment reset

### Run the coordination loop

At each step, the loop identifies satellites that require a new task, refreshes their shared acquisition information and mode-specific bookkeeping, and clears cached observations before selecting actions. It records new intents and their expected rewards; greedy modes can also displace a competing intent and request another decision. The action dictionary advances the constellation, rewards are accumulated across satellites, and the loop stops when the environment reports termination or truncation.

The current `policy` runs the actor defined above with randomly initialized weights and selects its highest-logit target/timing pair through a deterministic categorical distribution. It therefore demonstrates coordination with an untrained policy, rather than uniformly sampling random actions. Load trained module weights to evaluate a learned policy with the same loop.


In [15]:
while True:
    actions = {}
    current_time = env.simulator.sim_time

    old_intents = copy(intents)
    intents = {}

    intents_quality_old = copy(intents_quality)
    intents_quality = {}

    imaged_set = env.rewarder.data.imaged

    satellites_needing_retask = []
    for satellite in env.satellites:
        needs_retask = (
            satellite.name in info and info[satellite.name]["requires_retasking"]
        )
        exceeded_time_limit = (
            current_time - sat_last_decision_time[satellite.name] >= 500.0
        )
        intent_imaged = (
            old_intents[satellite.name] in imaged_set
            if old_intents[satellite.name] is not None
            else False
        ) and comm_mode not in ["incremental_intent", "greedy_incremental"]

        if needs_retask or exceeded_time_limit or intent_imaged:
            satellites_needing_retask.append(satellite.name)
            intents[satellite.name] = None
            intents_quality[satellite.name] = None
        else:
            intents[satellite.name] = old_intents[satellite.name]
            intents_quality[satellite.name] = intents_quality_old[satellite.name]

    while len(satellites_needing_retask) > 0:
        for satellite in env.satellites:
            if satellite.name in satellites_needing_retask:
                if comm_mode == "imaged":
                    satellite.imaged_set_shared = imaged_set

                if comm_mode == "intent":
                    satellite.imaged_set_shared = imaged_set
                    satellite.intent_set_shared = set(intents.values()) - {None}

                if comm_mode == "incremental_intent":
                    satellite.bookkeeping.delete_intended()
                    for tgt in env.rewarder.data.imaged:
                        satellite.bookkeeping.update(
                            tgt.id, tgt.highest_reward_obtained, current_time
                        )
                    for intent_quality in intents_quality.values():
                        if intent_quality is not None:
                            satellite.bookkeeping.update_intended(
                                intent_quality[0].id,
                                intent_quality[1],
                                intent_quality[2],
                            )

                if comm_mode == "greedy":
                    satellite.temp_filter_set = set()
                    satellite.imaged_set_shared = imaged_set

                if comm_mode == "greedy_incremental":
                    satellite.temp_filter_set = set()
                    satellite.bookkeeping.delete_intended()
                    for tgt in env.rewarder.data.imaged:
                        satellite.bookkeeping.update(
                            tgt.id, tgt.highest_reward_obtained, current_time
                        )

                satellite.observation_builder.obs_dict_cache = (
                    None  # Clear cache to force re-evaluation of the observation
                )

                sat_last_decision_time[satellite.name] = current_time
                action = policy(satellite)
                imaging_action = get_imaging_action(env, action)
                if (
                    comm_mode in ["greedy", "greedy_incremental"]
                    and imaging_action is not None
                ):
                    target_index, action_fraction = imaging_action
                    target_parsed = satellite.parse_target_selection(target_index)

                    while target_parsed in intents.values():
                        expected_absolute_reward = get_expected_absolute_reward(
                            env, satellite, target_index, action_fraction
                        )

                        intent_rewards = {
                            val[0].id: (key, val[1])
                            for key, val in intents_quality.items()
                            if val is not None
                        }

                        prev_data = intent_rewards.get(target_parsed.id)
                        prev_sat_name = prev_data[0]
                        prev_reward = prev_data[1]

                        if expected_absolute_reward > prev_reward:
                            # Assigns the new satellite's intent since it has a better expected reward than the previous satellite's intent
                            intents[satellite.name] = target_parsed
                            intents_quality[satellite.name] = (
                                target_parsed,
                                expected_absolute_reward,
                                get_time(env, action, satellite.name),
                            )

                            # Removes old satellites intent if the new satellite has a better expected reward
                            intents[prev_sat_name] = None
                            intents_quality[prev_sat_name] = None
                            actions[prev_sat_name] = None
                            satellites_needing_retask.append(prev_sat_name)
                            break
                        else:
                            satellite.temp_filter_set.add(target_parsed)
                            satellite.observation_builder.obs_dict_cache = None  # Clear cache to force re-evaluation of the observation
                            action = policy(satellite)

                            imaging_action = get_imaging_action(env, action)
                            if imaging_action is None:
                                break
                            target_index, action_fraction = imaging_action
                            target_parsed = satellite.parse_target_selection(
                                target_index
                            )

                actions[satellite.name] = action
                imaging_action = get_imaging_action(env, action)
                if imaging_action is not None:
                    target_index, action_fraction = imaging_action
                    target_parsed = satellite.parse_target_selection(target_index)
                    intents[satellite.name] = target_parsed
                    if comm_mode in [
                        "incremental_intent",
                        "greedy",
                        "greedy_incremental",
                    ]:
                        expected_absolute_reward = get_expected_absolute_reward(
                            env, satellite, target_index, action_fraction
                        )
                        intents_quality[satellite.name] = (
                            target_parsed,
                            expected_absolute_reward,
                            get_time(env, action, satellite.name),
                        )

                satellites_needing_retask.remove(satellite.name)

    _, reward, terminated, truncated, info = env.step(actions)  # type: ignore

    reward_cumulative += sum(reward.values())

    if any(terminated.values()) or any(truncated.values()):
        break

2026-10-26 21:02:09,946 gym                            INFO       <0.00> === STARTING STEP ===
2026-10-26 21:02:09,948 sats.satellite.EO_0            INFO       <0.00> EO_0: target index 3 tasked
2026-10-26 21:02:09,948 sats.satellite.EO_0            INFO       <0.00> EO_0: Target(Māhta, IN) imaging timing: requested capture at 11.8
2026-10-26 21:02:09,949 sats.satellite.EO_0            INFO       <0.00> EO_0: Target(Māhta, IN) tasked for imaging
2026-10-26 21:02:09,950 sats.satellite.EO_0            INFO       <0.00> EO_0: Target(Māhta, IN) window enabled: 0.0 to 23.7
2026-10-26 21:02:09,950 sats.satellite.EO_0            INFO       <0.00> EO_0: setting timed terminal event at 23.7
2026-10-26 21:02:09,952 sats.satellite.EO_1            INFO       <0.00> EO_1: target index 0 tasked
2026-10-26 21:02:09,953 sats.satellite.EO_1            INFO       <0.00> EO_1: Target(San Sebastián de la Gomera, ES) imaging timing: requested capture at 430.7
2026-10-26 21:02:09,953 sats.satellite.EO_1   